# Karting notebook

September 21, 2025

Analysis of my Karting experiences!

## extract the session data from the larger dataset

In [ ]:
import json
import dateutil.parser
from datetime import datetime

def simple_slice_and_save():
    """
    Simple function that loads the specified file, slices it to the 
    specified timestamp range, and saves the smaller dataset to a new file.
    """
    # File paths
    input_file = "/Users/nathanverrill/cota-karting/sensor_log_files/SensorLogFiles_nathan-iphone_250918_18-30-37/2025-09-18_17_52_17_nathan-iphone_converted.json"
    output_file = "./sessions.json"
    
    # Load the data
    print(f"Loading data from {input_file}")
    with open(input_file, 'r') as f:
        data = json.load(f)
    
    print(f"Loaded {len(data)} records")
    
    # Define timestamp range (milliseconds since epoch)
    start_timestamp_ms =    1758236521004        # Your specified start timestamp
    end_timestamp_ms =      1758236760807    # Your specified end timestamp
    

    # Convert milliseconds to seconds for comparison (if needed)
    start_timestamp_sec = start_timestamp_ms / 1000
    end_timestamp_sec = end_timestamp_ms / 1000
    
    # For reference, convert to human-readable format
    start_readable = datetime.fromtimestamp(start_timestamp_sec)
    end_readable = datetime.fromtimestamp(end_timestamp_sec)
    print(f"Filtering data from {start_readable} to {end_readable}")
    
    # Slice the data using loggingTime
    sliced_data = []
    time_field = 'loggingTime'
    
    for record in data:
        if time_field in record:
            try:
                # Parse the ISO format timestamp
                log_time_str = record[time_field]
                log_time = dateutil.parser.isoparse(log_time_str)
                
                # Convert to timestamp in milliseconds since epoch
                log_timestamp_ms = int(log_time.timestamp() * 1000)
                
                # Check if the timestamp is within our range
                if start_timestamp_ms <= log_timestamp_ms <= end_timestamp_ms:
                    sliced_data.append(record)
            except Exception as e:
                # Skip records with invalid timestamps
                continue
    
    print(f"Sliced data contains {len(sliced_data)} records")
    
    # Save the sliced data
    with open(output_file, 'w') as f:
        json.dump(sliced_data, f)
    
    print(f"Sliced data saved to {output_file}")

if __name__ == "__main__":
    simple_slice_and_save()

## interpolate data for smoother lines

The location data is noisy, so I want to smooth it into lines.

In [ ]:
import json
import pandas as pd
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
import os

# File paths
INPUT_FILE = "./sessions.json"
OUTPUT_DIR = os.path.dirname(INPUT_FILE)
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "./sessions_interpolated.json")
VISUALIZATION_FILE = os.path.join(OUTPUT_DIR, "gps_visualization.png")

def slice_data_by_timestamp(data_list, start_timestamp, end_timestamp, timestamp_field='locationTimestamp_since1970'):
    """
    Slices the dataset based on actual timestamp values rather than time tuples
    """
    df = pd.DataFrame(data_list)
    

    
    # Make sure the timestamp field exists
    if timestamp_field not in df.columns:
        print(f"Timestamp field '{timestamp_field}' not found.")
        return data_list
    
    # Filter the data
    filtered_df = df[(df[timestamp_field] >= start_timestamp) & 
                     (df[timestamp_field] <= end_timestamp)]
    
    if len(filtered_df) == 0:
        print(f"No data found in the specified time range: {datetime.fromtimestamp(start_timestamp)} to {datetime.fromtimestamp(end_timestamp)}")
        return []
    
    print(f"Found {len(filtered_df)} records in the specified time range")
    print(f"From {datetime.fromtimestamp(filtered_df[timestamp_field].min())} to {datetime.fromtimestamp(filtered_df[timestamp_field].max())}")
    
    return filtered_df.to_dict('records')

def interpolate_gps_path(data_list, points_between=5, method='linear', timestamp_field='locationTimestamp_since1970'):
    """
    Interpolates GPS data to create more points along a smooth curve
    """
    df = pd.DataFrame(data_list)
    
    # Skip if not enough points for interpolation
    if len(df) < 3:
        print("Not enough points for interpolation")
        return data_list
    
    # Check if we have GPS data
    if 'locationLatitude' not in df.columns or 'locationLongitude' not in df.columns:
        print("GPS data not found in the dataset")
        return data_list
        
    timestamps = df[timestamp_field].values
    lats = df['locationLatitude'].values
    lngs = df['locationLongitude'].values
    
    # Handle duplicate timestamps
    if len(timestamps) > 1:
        unique_timestamps = np.unique(timestamps)
        if len(unique_timestamps) < len(timestamps):
            print(f"Warning: Found {len(timestamps) - len(unique_timestamps)} duplicate timestamps")
            print("Creating adjusted timestamps for interpolation...")
            
            # Create new timestamps that are strictly increasing
            adjusted_timestamps = np.zeros_like(timestamps)
            adjusted_timestamps[0] = timestamps[0]
            
            for i in range(1, len(timestamps)):
                adjusted_timestamps[i] = max(timestamps[i], adjusted_timestamps[i-1] + 0.001)
            
            timestamps = adjusted_timestamps
    
    # Create new timestamps with more points
    new_timestamps = np.linspace(timestamps.min(), timestamps.max(), 
                               len(timestamps) * points_between + 1)
    
    # Choose interpolation method
    try:
        if method == 'cubic' and len(timestamps) > 3:
            # Cubic interpolation
            lat_interp = interp1d(timestamps, lats, kind='cubic', bounds_error=False, fill_value="extrapolate")
            lng_interp = interp1d(timestamps, lngs, kind='cubic', bounds_error=False, fill_value="extrapolate")
        else:
            # Linear interpolation (safe default)
            lat_interp = interp1d(timestamps, lats, kind='linear', bounds_error=False, fill_value="extrapolate")
            lng_interp = interp1d(timestamps, lngs, kind='linear', bounds_error=False, fill_value="extrapolate")
    except Exception as e:
        print(f"Interpolation error: {e}. Using linear interpolation.")
        lat_interp = interp1d(timestamps, lats, kind='linear', bounds_error=False, fill_value="extrapolate")
        lng_interp = interp1d(timestamps, lngs, kind='linear', bounds_error=False, fill_value="extrapolate")
    
    # Create new coordinates
    new_lats = lat_interp(new_timestamps)
    new_lngs = lng_interp(new_timestamps)
    
    # Create new dataframe with interpolated points
    result = []
    original_indices = set(range(0, len(new_timestamps), points_between))
    
    for i, (ts, lat, lng) in enumerate(zip(new_timestamps, new_lats, new_lngs)):
        if i in original_indices and i // points_between < len(df):
            # This is an original point
            record = df.iloc[i // points_between].to_dict()
            record['interpolated'] = False
            result.append(record)
        else:
            # This is an interpolated point
            # Find the nearest original point
            nearest_idx = min(int(i / points_between), len(df) - 1)
            new_record = df.iloc[nearest_idx].copy().to_dict()
            
            # Update with interpolated values
            new_record['locationLatitude'] = float(lat)
            new_record['locationLongitude'] = float(lng)
            new_record['interpolated'] = True
            new_record[timestamp_field] = float(ts)
            
            result.append(new_record)
    
    print(f"Interpolation complete: {len(data_list)} original points expanded to {len(result)} points")
    return result

def visualize_gps_data(data_list, output_file=None, show_interpolated=True):
    """
    Creates a visualization of the GPS data
    """
    df = pd.DataFrame(data_list)
    
    if 'interpolated' in df.columns:
        original_df = df[~df['interpolated']]
        interp_df = df[df['interpolated']]
        
        plt.figure(figsize=(12, 10))
        
        # Plot original points
        plt.plot(original_df['locationLongitude'], original_df['locationLatitude'], 
                'o', color='blue', markersize=6, label='Original GPS points')
        
        # Plot all points as a path
        plt.plot(df['locationLongitude'], df['locationLatitude'], 
                '-', color='red', linewidth=2, alpha=0.7, label='Interpolated path')
        
        if show_interpolated:
            # Plot interpolated points
            plt.plot(interp_df['locationLongitude'], interp_df['locationLatitude'], 
                    '.', color='green', markersize=3, alpha=0.5, label='Interpolated points')
    else:
        plt.figure(figsize=(12, 10))
        plt.plot(df['locationLongitude'], df['locationLatitude'], 
                'o-', color='blue', markersize=6, label='GPS points')
    
    plt.title('GPS Path Visualization')
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    
    if output_file:
        plt.savefig(output_file, dpi=300)
        print(f"Visualization saved to {output_file}")
    else:
        plt.show()

def process_karting_data():
    """
    Main function to process the karting data
    """
    print(f"Processing file: {INPUT_FILE}")
    
    # Load data
    with open(INPUT_FILE, 'r') as f:
        data = json.load(f)
    
    print(f"Loaded {len(data)} records")
    
    # Based on your timestamp analysis:
    # We'll use 11:01:30 PM UTC through 11:05:45 PM UTC
    # But first, let's convert these to actual timestamps in your data
    
    # Your data is from 2025-09-18 17:52:17 to 18:15:47 local time
    # Let's slice a portion from this range - for example, a 4-minute segment from 17:55:00 to 17:59:00
    
    # Convert to timestamp
    start_time = datetime(2025, 9, 18, 17, 55, 0).timestamp()
    end_time = datetime(2025, 9, 18, 17, 59, 0).timestamp()
    
    print(f"Slicing data from {datetime.fromtimestamp(start_time)} to {datetime.fromtimestamp(end_time)}")
    
    # Slice data
    sliced_data = slice_data_by_timestamp(
        data, 
        start_time,
        end_time,
        timestamp_field='locationTimestamp_since1970'
    )
    
    if not sliced_data:
        print("No data found in the specified time range. Using all data instead.")
        sliced_data = data
    
    # Interpolate to create smoother path
    print("Interpolating GPS path...")
    interpolated_data = interpolate_gps_path(
        sliced_data,
        points_between=10,  # Increase for smoother curves
        method='linear',  # Using linear for stability
        timestamp_field='locationTimestamp_since1970'
    )
    
    # Save processed data
    with open(OUTPUT_FILE, 'w') as f:
        json.dump(interpolated_data, f, indent=2)
    
    print(f"Processed data saved to {OUTPUT_FILE}")
    
    # Visualize the result
    visualize_gps_data(interpolated_data, VISUALIZATION_FILE)
    
    print("Processing complete!")

if __name__ == "__main__":
    process_karting_data()

## label laps and sectors

In [ ]:
import json
from datetime import datetime

def assign_laps_by_timestamps():
    """
    Assign lap numbers based on the specific timestamps you provided
    """
    # File paths
    input_file = "./sessions_interpolated.json"
    output_file = "./sessions_laps.json"
    
    # Load the data
    print(f"Loading data from {input_file}")
    with open(input_file, 'r') as f:
        data = json.load(f)
    
    print(f"Loaded {len(data)} records")
    
    # Define lap start timestamps (corrected)
    lap_timestamps = [
        1758236521053,  # Lap 1 start
        1758236600530,  # Lap 2 start
        1758236680816,    # Lap 3 start
    ]
    
    # Print human-readable versions of the timestamps for verification
    for i, ts in enumerate(lap_timestamps):
        # Convert milliseconds to seconds for datetime.fromtimestamp
        ts_seconds = ts / 1000
        print(f"Lap {i+1} starts at: {datetime.fromtimestamp(ts_seconds)} (timestamp: {ts})")
    
    # Verify timestamps are in chronological order
    if all(lap_timestamps[i] < lap_timestamps[i+1] for i in range(len(lap_timestamps)-1)):
        print("✓ Timestamps are in correct chronological order")
    else:
        print("⚠ Warning: Timestamps are not in chronological order")
    
    # Find the record indices closest to each timestamp
    timestamp_field = 'locationTimestamp_since1970'
    lap_start_indices = []
    
    for lap_num, target_time in enumerate(lap_timestamps, 1):
        closest_idx = 0
        closest_diff = float('inf')
        
        for i, record in enumerate(data):
            if timestamp_field in record:
                time_diff = abs(record[timestamp_field] - target_time)
                if time_diff < closest_diff:
                    closest_diff = time_diff
                    closest_idx = i
        
        lap_start_indices.append(closest_idx)
        print(f"Lap {lap_num} start found at record index {closest_idx} (time difference: {closest_diff:.6f}s)")
    
    # Add an end index (the length of the data) to simplify the loop
    lap_start_indices.append(len(data))
    
    # Assign lap numbers based on indices
    for i, record in enumerate(data):
        # Find which lap section this record belongs to
        lap_num = 1
        for j in range(len(lap_start_indices) - 1):
            if lap_start_indices[j] <= i < lap_start_indices[j + 1]:
                lap_num = j + 1
                break
        
        record['lap'] = lap_num
    
    # Save the updated data
    with open(output_file, 'w') as f:
        json.dump(data, f, indent=2)
    
    print(f"Data with lap numbers saved to {output_file}")
    
    # Print lap statistics
    lap_counts = {}
    for record in data:
        lap_num = record.get('lap', 1)
        lap_counts[lap_num] = lap_counts.get(lap_num, 0) + 1
    
    print("\nLap statistics:")
    for lap_num, count in sorted(lap_counts.items()):
        print(f"Lap {lap_num}: {count} records ({count/len(data)*100:.1f}%)")
    
    # Calculate time duration for each lap
    if len(data) > 0 and timestamp_field in data[0]:
        lap_times = {}
        for lap_num in range(1, len(lap_timestamps)+1):
            lap_records = [r for r in data if r.get('lap') == lap_num]
            if lap_records:
                start_time = min(r[timestamp_field] for r in lap_records if timestamp_field in r)
                end_time = max(r[timestamp_field] for r in lap_records if timestamp_field in r)
                duration = end_time - start_time
                lap_times[lap_num] = duration
        
        print("\nLap durations:")
        for lap_num, duration in sorted(lap_times.items()):
            print(f"Lap {lap_num}: {duration:.2f} seconds")

if __name__ == "__main__":
    assign_laps_by_timestamps()

## round location data

The GPS number of decimal places in iphone is overly optimistic and prone to noisy positions. Adding rounding and different geo representations

In [ ]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

# Load the JSON data
file_path = '/Users/nathanverrill/cota-karting/sensor_log_files/SensorLogFiles_nathan-iphone_250918_18-30-37/2025-09-18_17_52_17_nathan-iphone_converted.json'

# Check if the file exists
if not Path(file_path).exists():
    print(f"File not found: {file_path}")
else:
    # Read the JSON data
    with open(file_path, 'r') as f:
        data = json.load(f)
    
    # Convert to DataFrame if it's a list of records, or create a DataFrame with a single row if it's a single record
    if isinstance(data, list):
        df = pd.DataFrame(data)
    else:
        df = pd.DataFrame([data])
    
    # Create new columns with rounded coordinates
    df['lat_5'] = df['locationLatitude'].round(5)
    df['lng_5'] = df['locationLongitude'].round(5)
    
    # Display information about the data
    print(f"Loaded {len(df)} records")
    print("\nFirst few records with original and rounded coordinates:")
    display(df[['locationLatitude', 'lat_5', 'locationLongitude', 'lng_5']].head())
    
    # Calculate the difference to see how much was changed by rounding
    df['lat_diff'] = df['locationLatitude'] - df['lat_5']
    df['lng_diff'] = df['locationLongitude'] - df['lng_5']
    
    print("\nMaximum difference due to rounding:")
    print(f"Latitude max difference: {df['lat_diff'].abs().max():.8f}")
    print(f"Longitude max difference: {df['lng_diff'].abs().max():.8f}")
    
    # Save the modified data back to a new JSON file
    output_path = file_path.replace('.json', '_rounded.json')
    
    # Convert DataFrame back to dictionary or list format
    if len(df) == 1 and isinstance(data, dict):
        output_data = df.iloc[0].to_dict()
    else:
        output_data = df.to_dict(orient='records')
    
    with open(output_path, 'w') as f:
        json.dump(output_data, f, indent=2)
    
    print(f"\nSaved rounded data to: {output_path}")

In [ ]:
import numpy as np
import pandas as pd
from scipy.interpolate import CubicSpline, interp1d
import matplotlib.pyplot as plt
from datetime import datetime
import json

def interpolate_gps_path(data_list, points_between=5, method='cubic', timestamp_field='locationTimestamp_since1970'):
    """
    Interpolates GPS data to create more points along a smooth curve
    
    Args:
        data_list: List of dictionaries containing sensor data
        points_between: Number of points to insert between each original point
        method: Interpolation method ('cubic', 'spline', 'linear')
        timestamp_field: The specific field to use for timestamps
        
    Returns:
        List of dictionaries with original and interpolated points
    """
    df = pd.DataFrame(data_list)
    
    # Skip if not enough points for interpolation
    if len(df) < 3:
        return data_list
    
    # Check if we have GPS data and the specified timestamp field
    if 'locationLatitude' not in df.columns or 'locationLongitude' not in df.columns:
        print("GPS data not found in the dataset")
        return data_list
        
    if timestamp_field not in df.columns:
        print(f"Timestamp field '{timestamp_field}' not found. Available timestamp fields:")
        timestamp_candidates = [col for col in df.columns if any(x in col.lower() for x in ['time', 'timestamp'])]
        print("\n".join(timestamp_candidates))
        print("Using row index as fallback")
        timestamps = np.array(range(len(df)), dtype=float)
    else:
        timestamps = df[timestamp_field].values
    
    lats = df['locationLatitude'].values
    lngs = df['locationLongitude'].values
    
    # Ensure timestamps are strictly increasing
    if len(timestamps) > 1:
        for i in range(1, len(timestamps)):
            if timestamps[i] <= timestamps[i-1]:
                # If duplicate timestamp, add a small increment
                timestamps[i] = timestamps[i-1] + 0.001  # Add 1ms
    
    # Create new timestamps with more points
    new_timestamps = np.linspace(timestamps.min(), timestamps.max(), 
                               len(timestamps) * points_between + 1)
    
    # Choose interpolation method
    try:
        if method == 'cubic' and len(timestamps) > 3:
            # Cubic interpolation - smooth but can overshoot
            lat_interp = interp1d(timestamps, lats, kind='cubic', bounds_error=False, fill_value="extrapolate")
            lng_interp = interp1d(timestamps, lngs, kind='cubic', bounds_error=False, fill_value="extrapolate")
            
        elif method == 'spline' and len(timestamps) > 3:
            # Cubic spline - even smoother with better curve characteristics
            lat_cs = CubicSpline(timestamps, lats)
            lng_cs = CubicSpline(timestamps, lngs)
            lat_interp = lambda x: lat_cs(x)
            lng_interp = lambda x: lng_cs(x)
            
        else:
            # Linear interpolation - safe fallback, no overshooting
            lat_interp = interp1d(timestamps, lats, kind='linear', bounds_error=False, fill_value="extrapolate")
            lng_interp = interp1d(timestamps, lngs, kind='linear', bounds_error=False, fill_value="extrapolate")
    except Exception as e:
        print(f"Interpolation error: {e}. Falling back to linear.")
        lat_interp = interp1d(timestamps, lats, kind='linear', bounds_error=False, fill_value="extrapolate")
        lng_interp = interp1d(timestamps, lngs, kind='linear', bounds_error=False, fill_value="extrapolate")
    
    # Create new coordinates
    new_lats = lat_interp(new_timestamps)
    new_lngs = lng_interp(new_timestamps)
    
    # Create new dataframe with interpolated points
    result = []
    for i, (ts, lat, lng) in enumerate(zip(new_timestamps, new_lats, new_lngs)):
        # Find nearest original point for other values
        if i % points_between == 0 and i // points_between < len(df):
            # This is an original point
            record = df.iloc[i // points_between].to_dict()
            result.append(record)
        else:
            # This is an interpolated point
            # Start with the nearest original point's data
            nearest_idx = min(int(i / points_between), len(df) - 1)
            new_record = df.iloc[nearest_idx].to_dict()
            
            # Update with interpolated values
            new_record['locationLatitude'] = float(lat)
            new_record['locationLongitude'] = float(lng)
            new_record['interpolated'] = True  # Mark as interpolated
            
            # Update the timestamp field we used for interpolation
            new_record[timestamp_field] = float(ts)
            
            # If loggingTime exists, update it based on the new timestamp
            if 'loggingTime' in new_record and timestamp_field == 'locationTimestamp_since1970':
                try:
                    dt = datetime.fromtimestamp(ts)
                    new_record['loggingTime'] = dt.isoformat(timespec='milliseconds')
                except:
                    pass  # Keep original if conversion fails
            
            result.append(new_record)
    
    return result


# Load your data
with open('./sessions.json', 'r') as f:
    data = json.load(f)

# Interpolate the data to create a smoother path
# Explicitly use locationTimestamp_since1970 for GPS data
interpolated_data = interpolate_gps_path(
    data, 
    points_between=5, 
    method='linear',  # Start with linear which is more stable
    timestamp_field='locationTimestamp_since1970'
)

# Save interpolated data
with open('./sessions_interpolated.json', 'w') as f:
    json.dump(interpolated_data, f, indent=2)

## slim the data

There's a lot of data I don't need -- so trimming it by time

In [ ]:
import json
import dateutil.parser
from datetime import datetime

def simple_slice_and_save():
    """
    Simple function that loads the specified file, slices it to the 
    specified timestamp range, and saves the smaller dataset to a new file.
    """
    # File paths
    input_file = "./sessions.json"
    output_file = "./sessions_interpolated.json"
    
    # Load the data
    print(f"Loading data from {input_file}")
    with open(input_file, 'r') as f:
        data = json.load(f)
    
    print(f"Loaded {len(data)} records")
    
    # Define timestamp range (milliseconds since epoch)
    start_timestamp_ms = 1758236522000  # Your specified start timestamp
    end_timestamp_ms = 1758236750000    # Your specified end timestamp
    
    # Convert milliseconds to seconds for comparison (if needed)
    start_timestamp_sec = start_timestamp_ms / 1000
    end_timestamp_sec = end_timestamp_ms / 1000
    
    # For reference, convert to human-readable format
    start_readable = datetime.fromtimestamp(start_timestamp_sec)
    end_readable = datetime.fromtimestamp(end_timestamp_sec)
    print(f"Filtering data from {start_readable} to {end_readable}")
    
    # Slice the data using loggingTime
    sliced_data = []
    time_field = 'loggingTime'
    
    for record in data:
        if time_field in record:
            try:
                # Parse the ISO format timestamp
                log_time_str = record[time_field]
                log_time = dateutil.parser.isoparse(log_time_str)
                
                # Convert to timestamp in milliseconds since epoch
                log_timestamp_ms = int(log_time.timestamp() * 1000)
                
                # Check if the timestamp is within our range
                if start_timestamp_ms <= log_timestamp_ms <= end_timestamp_ms:
                    sliced_data.append(record)
            except Exception as e:
                # Skip records with invalid timestamps
                continue
    
    print(f"Sliced data contains {len(sliced_data)} records")
    
    # Save the sliced data
    with open(output_file, 'w') as f:
        json.dump(sliced_data, f)
    
    print(f"Sliced data saved to {output_file}")

if __name__ == "__main__":
    simple_slice_and_save()

## Sector labeling

In [ ]:
import json
from shapely.geometry import Point, Polygon

def load_sectors(geojson_file):
    """Load sector definitions from GeoJSON file"""
    with open(geojson_file, 'r') as f:
        geojson = json.load(f)
    
    sectors = {}
    for feature in geojson['features']:
        sector_id = feature['properties']['Sector']
        coords = feature['geometry']['coordinates'][0]  # Get polygon coordinates
        sectors[sector_id] = Polygon(coords)
    
    return sectors

def label_time_series(time_series_data, sectors):
    """Label time series data based on sectors"""
    labeled_data = []
    current_sector = None  # Start with no sector
    
    for record in time_series_data:
        # Extract location data
        lat = record.get('locationLatitude')
        lon = record.get('locationLongitude')
        
        if lat is not None and lon is not None:
            point = Point(lon, lat)  # GeoJSON uses (lon, lat) order
            
            # Check if point is in any sector
            for sector_id, polygon in sectors.items():
                if polygon.contains(point):
                    if current_sector != sector_id:
                        # Point has entered a new sector
                        current_sector = sector_id
                    break
        
        # Apply the current sector label (stays the same until a new sector is entered)
        if current_sector is not None:
            record['sectorLabel'] = current_sector
        else:
            record['sectorLabel'] = 0  # Default when not in any sector yet
        
        labeled_data.append(record)
    
    return labeled_data

def process_file(input_file, geojson_file, output_file=None):
    """Process a single time series file"""
    # Load the time series data
    with open(input_file, 'r') as f:
        data = json.load(f)
    
    # Convert to list if it's a single record
    if isinstance(data, dict):
        time_series_data = [data]
    else:
        time_series_data = data
    
    # Load sector polygons
    sectors = load_sectors(geojson_file)
    
    # Label the time series data
    labeled_data = label_time_series(time_series_data, sectors)
    
    # Determine output file name if not provided
    if output_file is None:
        output_file = input_file.replace('.json', '_labeled.json')
    
    # Save the labeled data
    with open(output_file, 'w') as f:
        json.dump(labeled_data, f, indent=2)
    
    print(f"Labeled data saved to {output_file}")
    
    return labeled_data


# Example usage
input_file = "/Users/nathanverrill/cota-karting/2025-09-21_16_48_05_nathan-iphone_converted.json"

geojson_file = "sectors.geojson"

process_file(input_file, geojson_file)



## Next data proceessing

In [ ]:
import polars as pl
import geopandas as gpd
from shapely.geometry import Point, shape
import json

def add_sector_labels(df, geojson_path):
    # Load the sectors GeoJSON
    with open(geojson_path, 'r') as f:
        sectors_data = json.load(f)
    
    # Create list of sector polygons with their names
    sectors = []
    for feature in sectors_data['features']:
        name = feature['properties']['Name']
        geometry = shape(feature['geometry'])  # Convert to shapely geometry
        sectors.append((name, geometry))
    
    # Create a function to find the sector for each point
    def find_sector(lon, lat):
        point = Point(lon, lat)
        for name, polygon in sectors:
            if polygon.contains(point):
                # Extract just the number from "Sector X"
                return int(name.split()[1]) if "Sector" in name else 0
        return 0  # No Sector
    
    # Extract coordinates
    temp_df = {
        'locationLongitude': df['locationLongitude'].to_list(),
        'locationLatitude': df['locationLatitude'].to_list()
    }
    
    # Calculate raw sectors for all points
    raw_sectors = [find_sector(lon, lat) for lon, lat in zip(
        temp_df['locationLongitude'], 
        temp_df['locationLatitude']
    )]
    
    # Post-process: handle sequential values and carry previous sector value
    processed_sectors = []
    prev_sector = 0
    
    for i, sector in enumerate(raw_sectors):
        # If current point is in a sector, use that
        if sector != 0:
            processed_sectors.append(sector)
            prev_sector = sector
        # If current point is not in a sector (0)
        else:
            # Look ahead to see if we quickly return to a sector
            look_ahead = 5  # How many points to look ahead
            future_sectors = raw_sectors[i+1:i+look_ahead+1] if i+1 < len(raw_sectors) else []
            
            # If we quickly return to the same sector, use previous sector
            if future_sectors and any(s == prev_sector for s in future_sectors):
                next_valid_idx = next((j for j, s in enumerate(future_sectors) if s != 0), None)
                if next_valid_idx is not None and next_valid_idx <= 2:  # Only if we return within 2 points
                    processed_sectors.append(prev_sector)
                else:
                    processed_sectors.append(0)  # Use 0 for longer gaps
            else:
                processed_sectors.append(0)
    
    # Ensure wrapping from 3 to 0 (if needed)
    final_sectors = []
    for i in range(len(processed_sectors)):
        if i > 0 and processed_sectors[i-1] == 3 and processed_sectors[i] == 0:
            # Check if this is just a brief dropout or a genuine wrap
            look_ahead = min(5, len(processed_sectors) - i)
            if any(s > 0 for s in processed_sectors[i:i+look_ahead]):
                final_sectors.append(processed_sectors[i-1])  # Keep as 3
            else:
                final_sectors.append(0)  # Genuine wrap to 0
        else:
            final_sectors.append(processed_sectors[i])
    
    # Create a new dataframe with the added sector column
    return df.with_columns([
        pl.Series(name="sector", values=final_sectors)
    ])

# Example usage
df_sectors = add_sector_labels(df, 'sectors.geojson')
df_sectors.write_csv('kart_with_sectors.csv')

# Show the results
if "sector" in df_sectors.columns:
    # Print a summary showing sequences of sectors
    sectors_list = df_sectors["sector"].to_list()
    print("Sector sequence pattern:")
    current = sectors_list[0]
    count = 1
    pattern = []
    for s in sectors_list[1:]:
        if s == current:
            count += 1
        else:
            pattern.append(f"{current}×{count}")
            current = s
            count = 1
    pattern.append(f"{current}×{count}")
    print(", ".join(pattern))
    
    print("\nFirst few rows:")
    print(df_sectors.select(["locationLongitude", "locationLatitude", "sector"]).head())
else:
    print("Sector column was not created properly")

## Session 

Calculate the session based on a start finish line for a flying lap

In [ ]:
import polars as pl

# Check the data type and time range in the DataFrame
print(f"loggingTime column type: {df.schema['loggingTime']}")
print(f"Total records: {df.shape[0]}")

# Get the minimum and maximum timestamps
if df.shape[0] > 0:
    min_time = df["loggingTime"].min()
    max_time = df["loggingTime"].max()
    print(f"Data time range: {min_time} to {max_time}")
    
    # Show a few sample timestamps for better understanding
    print("\nSample timestamps:")
    print(df.select("loggingTime").sample(5, seed=42))
    
    # If you have other columns that might help identify the session
    # For example, if there's a 'session_id' or similar column
    if 'session_id' in df.columns:
        print("\nUnique session IDs:")
        print(df.select('session_id').unique().sort('session_id'))
    
    # Let's look at the distribution of timestamps
    # Create time bins to see where the data is concentrated
    print("\nData distribution by hour:")
    hour_counts = df.group_by(pl.col("loggingTime").dt.hour()).count().sort("loggingTime")
    print(hour_counts)
    
    # Based on the actual data range, suggest filter times
    print("\nSuggested filter times based on your data:")
    print(f"start_time = datetime({min_time.year}, {min_time.month}, {min_time.day}, " 
          f"{min_time.hour}, {min_time.minute}, {min_time.second}, tzinfo=timezone.utc)")
    print(f"end_time = datetime({max_time.year}, {max_time.month}, {max_time.day}, " 
          f"{max_time.hour}, {max_time.minute}, {max_time.second}, tzinfo=timezone.utc)")

## Sectors

Label each datapoint with the sector it falls with in.

Sectors are defined with a geojson file that I created on the geojson.io website.

In [ ]:
import polars as pl
import geopandas as gpd
from shapely.geometry import Point, shape
import json

def add_sector_labels(df, geojson_path):
    # Load the sectors GeoJSON
    with open(geojson_path, 'r') as f:
        sectors_data = json.load(f)
    
    # Debug - inspect coordinate ranges in your data
    lon_min = float(df['locationLongitude'].min())
    lon_max = float(df['locationLongitude'].max())
    lat_min = float(df['locationLatitude'].min())
    lat_max = float(df['locationLatitude'].max())
    print(f"Data coordinate ranges: Longitude [{lon_min}, {lon_max}], Latitude [{lat_min}, {lat_max}]")
    
    # Create list of sector polygons with their names
    sectors = []
    for feature in sectors_data['features']:
        name = feature['properties']['Name']
        geometry = shape(feature['geometry'])
        
        # Print bounds of each sector
        bounds = geometry.bounds
        print(f"{name} bounds: Longitude [{bounds[0]}, {bounds[2]}], Latitude [{bounds[1]}, {bounds[3]}]")
        
        # Add a larger buffer to capture points near boundaries
        buffered_geometry = geometry.buffer(0.0001)  # Increased buffer size
        sectors.append((name, geometry, buffered_geometry))
    
    print(f"Loaded {len(sectors)} sector polygons")
    
    # Convert string coordinates to float if they're not already
    df = df.with_columns([
        pl.col('locationLongitude').cast(pl.Float64),
        pl.col('locationLatitude').cast(pl.Float64)
    ])
    
    # Create a function to find the sector for each point
    def find_sector(lon, lat):
        point = Point(lon, lat)
        
        # First try exact containment
        for name, polygon, _ in sectors:
            if polygon.contains(point):
                return int(name.split()[1]) if "Sector" in name else 0
        
        # Then try with buffer
        for name, _, buffered in sectors:
            if buffered.contains(point):
                return int(name.split()[1]) if "Sector" in name else 0
                
        return 0  # No Sector
    
    # Extract coordinates
    temp_df = {
        'locationLongitude': df['locationLongitude'].to_list(),
        'locationLatitude': df['locationLatitude'].to_list()
    }
    
    # Calculate raw sectors for all points
    raw_sectors = [find_sector(lon, lat) for lon, lat in zip(
        temp_df['locationLongitude'], 
        temp_df['locationLatitude']
    )]
    
    # Count sector frequencies for debugging
    from collections import Counter
    sector_freq = Counter(raw_sectors)
    print("Raw sector assignments:", dict(sector_freq))
    
    # Post-process to create smoother sector transitions
    final_sectors = []
    prev_sector = 0
    window_size = 5
    
    for i in range(len(raw_sectors)):
        current = raw_sectors[i]
        
        # If point is in a sector, use that
        if current != 0:
            final_sectors.append(current)
            prev_sector = current
            continue
            
        # If point is not in a sector (0)
        # Look at a window of points before and after
        start = max(0, i - window_size)
        end = min(len(raw_sectors), i + window_size + 1)
        window = raw_sectors[start:i] + raw_sectors[i+1:end]
        
        # Filter out zeros
        window = [s for s in window if s != 0]
        
        if not window:
            # If no non-zero sectors in window, use 0
            final_sectors.append(0)
        else:
            # Count occurrences of each sector in the window
            from collections import Counter
            counts = Counter(window)
            most_common = counts.most_common(1)[0][0]
            
            # Use most common nearby sector
            final_sectors.append(most_common)
            prev_sector = most_common
    
    # Create a new dataframe with the added sector column
    return df.with_columns([
        pl.Series(name="sector", values=final_sectors)
    ])

# Example usage
df_sectors = add_sector_labels(df, 'sectors.geojson')
df_sectors.write_csv('kart_with_sectors.csv')

# Show the results
if "sector" in df_sectors.columns:
    print(df_sectors.select(["locationLongitude", "locationLatitude", "sector"]).head())
    
    # Count how many points are in each sector - using Polars syntax
    sector_counts = df_sectors.filter(pl.col("sector") == 0).shape[0]
    print(f"\nPoints in sector 0: {sector_counts}")
    sector_counts = df_sectors.filter(pl.col("sector") == 1).shape[0]
    print(f"Points in sector 1: {sector_counts}")
    sector_counts = df_sectors.filter(pl.col("sector") == 2).shape[0]
    print(f"Points in sector 2: {sector_counts}")
    sector_counts = df_sectors.filter(pl.col("sector") == 3).shape[0]
    print(f"Points in sector 3: {sector_counts}")

In [ ]:
df_labeled.write_csv('kart_race_data_sectors.json')


## G-force

Calculate G-Force. Since the orientation of the phane changes, need to calculate it.

In [ ]:
import polars as pl
import numpy as np

def create_calculated_gforce_df(df: pl.DataFrame) -> pl.DataFrame:
    """
    Creates a new DataFrame with only the calculated total_g_force column.
    """
    # First, calculate the new column on the existing dataframe
    df_with_gforce = df.with_columns(
        (
            pl.col("motionUserAccelerationX").cast(pl.Float64)**2 +
            pl.col("motionUserAccelerationY").cast(pl.Float64)**2 +
            pl.col("motionUserAccelerationZ").cast(pl.Float64)**2
        ).sqrt().alias("total_g_force")
    )
    
    # Then, select only the new column from the new dataframe
    return df_with_gforce.select(pl.col("total_g_force"))


## Time since start

This function calculates a cumulative time_since_start column, which is the basis for all lap and sector timing analysis.

In [ ]:
import polars as pl

def calculate_time_since_start(df: pl.DataFrame) -> pl.DataFrame:
    """
    Calculates time in seconds since the start of the recording.
    """
    start_time = df.select(pl.col("loggingTime").min()).item()
    return df.with_columns(
        (pl.col("loggingTime") - start_time).alias("time_since_start")
    )

## Delta Time

This function calculates the time difference between your current lap and a reference (best) lap, allowing you to see if you are gaining or losing time. You'll need to define the best lap data separately.

In [ ]:
import polars as pl

def calculate_delta_time(df: pl.DataFrame, best_lap_df: pl.DataFrame) -> pl.DataFrame:
    """
    Calculates the time difference (delta) compared to a best lap.
    Requires aligning data based on distance or time.
    """
    # For a simple example, let's assume we align on a 'distance' column
    # You would typically resample or join to align the data points
    return df.join(best_lap_df, on="pedometerDistance", suffix="_best", how="left").with_columns(
        (pl.col("time_since_start") - pl.col("time_since_start_best")).alias("delta_time")
    )


## Speed and Velocity

This function calculates a smoothed speed and velocity from the location data to reduce GPS jitter.

In [ ]:
import polars as pl
import numpy as np

def calculate_smoothed_speed(df: pl.DataFrame, window_size: int = 5) -> pl.DataFrame:
    """
    Calculates a rolling average of location speed to smooth out data.
    """
    return df.with_columns(
        pl.col("locationSpeed").rolling_mean(window_size=window_size).alias("smoothed_speed")
    )


In [ ]:
df_with_gforce = create_calculated_gforce_df(df)


In [ ]:
df_with_gforce.head()